---
##  1: Mount Google Drive & Locate ConvFinQA Files


In [2]:
from google.colab import drive
import os
import re
import json
import zipfile
import pandas as pd
import numpy as np

# Mount Google Drive
drive.mount('/content/drive')

DRIVE_PROJECT_DIR = '/content/drive/MyDrive/Data Cleaning'

# Auto-detect folder location if placed in another directory
if not os.path.exists(DRIVE_PROJECT_DIR):
    print("Searching for ConvFinQA dataset across Google Drive...")
    for root, dirs, files in os.walk('/content/drive/MyDrive'):
        if any('convfinqa' in f.lower() for f in files):
            DRIVE_PROJECT_DIR = root
            print(f"✅ Found project folder at: {DRIVE_PROJECT_DIR}")
            break
else:
    print(f"✅ Drive mounted successfully! Working folder: {DRIVE_PROJECT_DIR}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Drive mounted successfully! Working folder: /content/drive/MyDrive/Data Cleaning


---
## 2: Configure Environment & Visualization Styles


In [3]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

print("✅ Visualization styling ready!")


✅ Visualization styling ready!


---
##  3: Define Cleaning & Markdown Transformation Functions


In [4]:
def clean_text(text: str) -> str:
    """Cleans Unicode artifacts, tokenized punctuation spacing, and trims strings."""
    if not isinstance(text, str):
        return ""
    text = text.replace('\xa0', ' ').replace('\u200b', '').replace('\ufeff', '')
    text = re.sub(r'\s+([,.:;?!%])', r'\1', text)
    text = re.sub(r'\$\s+(\d)', r'$\1', text)
    text = re.sub(r'\(\s+', '(', text)
    text = re.sub(r'\s+\)', ')', text)
    text = re.sub(r"\s+'s\b", "'s", text)
    text = re.sub(r'[ \t]+', ' ', text)
    return text.strip()

def clean_paragraphs(para_list):
    if not isinstance(para_list, list):
        return []
    cleaned = []
    for p in para_list:
        c = clean_text(p.get('text', '') if isinstance(p, dict) else p)
        if c and c not in ['.', ',', '-', ';', ':', '?', '!']:
            cleaned.append(c)
    return cleaned

def clean_table_matrix(table_matrix):
    if not isinstance(table_matrix, list):
        return []
    cleaned_rows = []
    for row in table_matrix:
        if isinstance(row, list):
            cleaned_row = [clean_text(str(cell)) if cell is not None else "" for cell in row]
            if any(cell != "" for cell in cleaned_row):
                cleaned_rows.append(cleaned_row)
    return cleaned_rows

def table_to_markdown(table_matrix):
    if not table_matrix or not isinstance(table_matrix, list):
        return ""
    num_cols = max(len(r) for r in table_matrix) if table_matrix else 0
    if num_cols == 0:
        return ""

    norm_matrix = [r + [""] * (num_cols - len(r)) for r in table_matrix]
    header = "| " + " | ".join(norm_matrix[0]) + " |"
    separator = "| " + " | ".join(["---"] * num_cols) + " |"
    rows = [header, separator]
    for r in norm_matrix[1:]:
        rows.append("| " + " | ".join(r) + " |")
    return "\n".join(rows)

def parse_filename(filename: str):
    if not isinstance(filename, str) or not filename:
        return {"company_ticker": "UNKNOWN", "report_year": "UNKNOWN", "page_num": "UNKNOWN"}
    parts = filename.replace('\\', '/').split('/')
    company = parts[0] if len(parts) > 0 else "UNKNOWN"
    year = parts[1] if len(parts) > 1 else "UNKNOWN"
    page_part = parts[2] if len(parts) > 2 else ""
    page_match = re.search(r'page_(\d+)', page_part)
    page_num = page_match.group(1) if page_match else page_part
    return {"company_ticker": company, "report_year": year, "page_num": page_num}

def detect_primary_operation(program_str: str) -> str:
    if not program_str or not isinstance(program_str, str):
        return "lookup"
    ops = re.findall(r'([a-zA-Z_]+)\(', program_str)
    if not ops:
        return "lookup"
    if len(ops) == 1:
        return ops[0].lower()
    return f"compound ({', '.join(sorted(set(ops)))})"

print("✅ Cleaning and transformation functions registered!")


✅ Cleaning and transformation functions registered!


---
##  4: Process & Flatten ConvFinQA Splits


In [6]:
# Find zip or unzipped files
zip_candidates = [
    os.path.join(DRIVE_PROJECT_DIR, 'ConvFinQA-20261013T090340Z-1-001.zip'),
    '/content/drive/MyDrive/ConvFinQA-20261013T090340Z-1-001.zip',
    '/content/ConvFinQA-20261013T090340Z-1-001.zip'
]

archive_path = None
for z in zip_candidates:
    if os.path.exists(z):
        archive_path = z
        break

train_raw, dev_raw, test_raw = None, None, None

if archive_path:
    print(f"📦 Loading dataset directly from archive: {archive_path}")
    with zipfile.ZipFile(archive_path, 'r') as z:
        train_raw = json.loads(z.read("ConvFinQA/data/data/train.json").decode('utf-8'))
        dev_raw = json.loads(z.read("ConvFinQA/data/data/dev.json").decode('utf-8'))
        test_raw = json.loads(z.read("ConvFinQA/data/data/test_private.json").decode('utf-8'))
else:
    print("📁 Checking unzipped folder paths...")
    train_path = os.path.join(DRIVE_PROJECT_DIR, 'ConvFinQA/data/data/train.json')
    dev_path = os.path.join(DRIVE_PROJECT_DIR, 'ConvFinQA/data/data/dev.json')
    test_path = os.path.join(DRIVE_PROJECT_DIR, 'ConvFinQA/data/data/test_private.json')

    if os.path.exists(train_path):
        with open(train_path, 'r', encoding='utf-8') as f:
            train_raw = json.load(f)
        with open(dev_path, 'r', encoding='utf-8') as f:
            dev_raw = json.load(f)
        with open(test_path, 'r', encoding='utf-8') as f:
            test_raw = json.load(f)
    else:
        raise FileNotFoundError(
            f"❌ Could not find ConvFinQA files! Please ensure that either:\n"
            f"1. The zip file exists in one of these paths: {zip_candidates}\n"
            f"2. Or the unzipped data exists at: {train_path}"
        )

def process_convfinqa_split(split_name, raw_items):
    turn_records, syn_records, doc_records = [], [], []
    for item in raw_items:
        doc_id = item.get('id', '')
        filename = item.get('filename', '')
        file_meta = parse_filename(filename)
        prob_type = "Double" if doc_id.startswith("Double_") or "qa_0" in item else "Single"

        raw_table = item.get('table_ori', []) or item.get('table', [])
        cleaned_table = clean_table_matrix(raw_table)
        table_rows = len(cleaned_table)
        table_cols = max([len(r) for r in cleaned_table]) if cleaned_table else 0
        table_md = table_to_markdown(cleaned_table)

        cleaned_pre = clean_paragraphs(item.get('pre_text', []))
        cleaned_post = clean_paragraphs(item.get('post_text', []))
        total_context_words = sum(len(p.split()) for p in cleaned_pre + cleaned_post)

        ann = item.get('annotation', {}) if isinstance(item.get('annotation'), dict) else {}
        dialogue_break = ann.get('dialogue_break', [])
        exe_ans_list = ann.get('exe_ans_list', [])
        turn_program = ann.get('turn_program', [])
        answer_list = ann.get('answer_list', [])
        total_turns = len(dialogue_break)

        for turn_idx, turn_q_raw in enumerate(dialogue_break):
            turn_q = clean_text(turn_q_raw)
            turn_id = f"{doc_id}_turn_{turn_idx}"
            turn_exe_ans = exe_ans_list[turn_idx] if turn_idx < len(exe_ans_list) else None
            turn_ans_repr = answer_list[turn_idx] if turn_idx < len(answer_list) else ""
            if turn_exe_ans is not None and not turn_ans_repr:
                turn_ans_repr = str(turn_exe_ans)

            turn_prog = turn_program[turn_idx] if turn_idx < len(turn_program) else ""
            if not turn_prog or '(' not in turn_prog:
                turn_type = "number_turn"
                op = "lookup"
            else:
                turn_type = "program_turn"
                op = detect_primary_operation(turn_prog)

            turn_records.append({
                "turn_id": turn_id,
                "doc_id": doc_id,
                "split": split_name,
                "turn_index": turn_idx,
                "total_turns": total_turns,
                "company_ticker": file_meta["company_ticker"],
                "report_year": file_meta["report_year"],
                "turn_type": turn_type,
                "operation": op,
                "question": turn_q,
                "answer": str(turn_ans_repr),
                "executed_answer": turn_exe_ans,
                "turn_program": turn_prog,
                "question_length_words": len(turn_q.split()),
                "has_table": 1 if table_rows > 0 else 0,
                "table_rows": table_rows,
                "table_cols": table_cols,
                "total_context_words": total_context_words
            })

        qa_candidates = []
        if 'qa' in item and isinstance(item['qa'], dict):
            qa_candidates.append(('single', f"{doc_id}_qa", item['qa']))
        if 'qa_0' in item and isinstance(item['qa_0'], dict):
            qa_candidates.append(('double_0', f"{doc_id}_qa_0", item['qa_0']))
        if 'qa_1' in item and isinstance(item['qa_1'], dict):
            qa_candidates.append(('double_1', f"{doc_id}_qa_1", item['qa_1']))

        for q_type, qa_id, qa_data in qa_candidates:
            q_text = clean_text(qa_data.get('question', ''))
            ans_text = clean_text(str(qa_data.get('answer', '')))
            exe_ans = qa_data.get('exe_ans', None)
            prog = clean_text(qa_data.get('program', ''))
            prog_re = clean_text(qa_data.get('program_re', ''))
            ops = ", ".join(sorted(set(re.findall(r'([a-zA-Z_]+)\(', prog or prog_re)))) or "lookup"

            syn_records.append({
                "qa_id": qa_id,
                "doc_id": doc_id,
                "split": split_name,
                "question_type": q_type,
                "company_ticker": file_meta["company_ticker"],
                "report_year": file_meta["report_year"],
                "question": q_text,
                "answer": ans_text,
                "executed_answer": exe_ans,
                "program": prog,
                "program_executable": prog_re,
                "operations_used": ops,
                "question_length_words": len(q_text.split()),
                "table_rows": table_rows,
                "table_cols": table_cols,
                "total_context_words": total_context_words
            })

        doc_records.append({
            "doc_id": doc_id,
            "split": split_name,
            "filename": filename,
            "company_ticker": file_meta["company_ticker"],
            "report_year": file_meta["report_year"],
            "page_num": file_meta["page_num"],
            "problem_type": prob_type,
            "num_dialogue_turns": total_turns,
            "table_rows": table_rows,
            "table_cols": table_cols,
            "total_context_words": total_context_words,
            "table_markdown": table_md
        })

    return turn_records, syn_records, doc_records

all_turns, all_syn, all_docs = [], [], []
for sname, sdata in [('train', train_raw), ('dev', dev_raw), ('test', test_raw)]:
    t_recs, s_recs, d_recs = process_convfinqa_split(sname, sdata)
    all_turns.extend(t_recs)
    all_syn.extend(s_recs)
    all_docs.extend(d_recs)

df_turns = pd.DataFrame(all_turns)
df_synthesis = pd.DataFrame(all_syn)
df_docs = pd.DataFrame(all_docs)

print(f"\n🎉 Total Cleaned Documents: {len(df_docs):,}")
print(f"🎉 Total Cleaned Dialogue Turns: {len(df_turns):,}")
print(f"🎉 Total Cleaned Synthesis QAs: {len(df_synthesis):,}")
display(df_turns.head(3))

📁 Checking unzipped folder paths...


FileNotFoundError: ❌ Could not find ConvFinQA files! Please ensure that either:
1. The zip file exists in one of these paths: ['/content/drive/MyDrive/Data Cleaning/ConvFinQA-20261013T090340Z-1-001.zip', '/content/drive/MyDrive/ConvFinQA-20261013T090340Z-1-001.zip', '/content/ConvFinQA-20261013T090340Z-1-001.zip']
2. Or the unzipped data exists at: /content/drive/MyDrive/Data Cleaning/ConvFinQA/data/data/train.json

---
##  5: Visualizations & Financial Reasoning Analytics


In [ ]:
# 1. Dialogue Turns Per Conversation
plt.figure(figsize=(10, 5))
sns.countplot(data=df_docs, x='num_dialogue_turns', palette='mako')
plt.title("Distribution of Conversation Depth (Turns per Financial Report)")
plt.xlabel("Number of Dialogue Turns")
plt.ylabel("Document Count")
plt.show()

# 2. Turn Type Distribution (Retrieval vs Reasoning)
plt.figure(figsize=(8, 5))
sns.countplot(data=df_turns, x='turn_type', palette='crest')
plt.title("Conversational Turn Types: Number Lookup vs Program Calculation")
plt.xlabel("Turn Type")
plt.ylabel("Turn Count")
plt.show()

# 3. Top Arithmetic Operations
plt.figure(figsize=(12, 6))
top_ops = df_turns['operation'].value_counts().head(8)
sns.barplot(x=top_ops.values, y=top_ops.index, palette='rocket')
plt.title("Most Frequent Operations in ConvFinQA Programs")
plt.xlabel("Occurrences")
plt.ylabel("Operation")
plt.show()

# 4. Top 15 Analyzed Companies
plt.figure(figsize=(12, 6))
top_companies = df_docs['company_ticker'].value_counts().head(15)
sns.barplot(x=top_companies.values, y=top_companies.index, palette='viridis')
plt.title("Top S&P 500 Companies Analyzed in ConvFinQA")
plt.xlabel("Report Count")
plt.ylabel("Company Ticker")
plt.show()

# 5. Question Word Count Distribution
plt.figure(figsize=(10, 5))
sns.histplot(df_turns['question_length_words'], bins=25, kde=True, color='purple')
plt.title("Dialogue Question Word Count Distribution")
plt.xlabel("Word Count")
plt.ylabel("Frequency")
plt.show()


---
##  6: Export Cleaned Datasets back to Google Drive


In [ ]:
turns_export_path = os.path.join(DRIVE_PROJECT_DIR, 'convfinqa_dialogue_turns_cleaned.csv')
syn_export_path = os.path.join(DRIVE_PROJECT_DIR, 'convfinqa_synthesis_qa_cleaned.csv')
docs_export_path = os.path.join(DRIVE_PROJECT_DIR, 'convfinqa_documents_cleaned.csv')

df_turns.to_csv(turns_export_path, index=False, encoding='utf-8')
df_synthesis.to_csv(syn_export_path, index=False, encoding='utf-8')
df_docs.to_csv(docs_export_path, index=False, encoding='utf-8')

print("🎉 Successfully exported cleaned datasets directly to Google Drive:")
print(f"👉 {turns_export_path}")
print(f"👉 {syn_export_path}")
print(f"👉 {docs_export_path}")

try:
    from google.colab import files
    print("\nDownloading Turns CSV to local machine...")
    files.download(turns_export_path)
except Exception as e:
    print(f"Note: Direct browser download skipped ({e}). File is stored safely in Google Drive.")
